In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [ ]:
class BatsManState(TypedDict):
    runs: int
    balls_played: int
    sixes: int
    fours: int

    strick_rate: float
    balls_per_boundary: float
    boundary_percent: float
    summary: str

In [ ]:
# in parallel workflows we cannot return the whole state from node to avoid conflict instead we have to return partial update

def calculate_sr(state: BatsManState):
    sr = ( state['runs'] / state['balls_played'] ) * 100

    return {'strick_rate': sr}              # partial update


def calculate_bpb(state: BatsManState):
    bpb = state['balls_played'] / (state['fours'] + state['sixes'])

    return {'balls_per_boundary': bpb}      # partial update


def calculate_boundary_percent(state: BatsManState):
    bp = ( ( (state['fours']*4) + (state['sixes']*6) ) / state['runs'] ) * 100

    return {'boundary_percent': bp}        # partial update


def summary(state: BatsManState):
    summary = f"""Strick Rate: {state['strick_rate']}
Balls Per Boundary: {state['balls_per_boundary']}
Boundary Percentage: {state['boundary_percent']}"""

    return {'summary': summary}            # partial update

In [ ]:
graph = StateGraph(BatsManState)


graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)


graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percent')
graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')


workflow = graph.compile()

In [ ]:
initial_state = {
    'runs': 100, 
    'balls_played': 50, 
    'sixes': 5, 
    'fours': 5
}

final_state = workflow.invoke(initial_state)

print(final_state)

{'runs': 100, 'balls_played': 50, 'sixes': 5, 'fours': 5, 'strick_rate': 200.0, 'balls_per_boundary': 5.0, 'boundary_percent': 50.0, 'summary': 'Strick Rate: 200.0\nBalls Per Boundary: 5.0\nBoundary Percentage: 50.0'}
